# 🛠️ Project Overview: Brute Force Forward Feature Selection

This exercise implements **Forward Selection**, a **greedy search algorithm** used for **Dimensionality Reduction** (Feature Selection). The goal is to establish an **order of feature importance** for a **RandomForest Regressor** by iteratively selecting the feature that yields the greatest reduction in **Mean Absolute Error (MAE)**.

---

## 🔍 Ex 1: Forward Feature Selection Methodology

| ML/Data Science Principle | Concept | Goal |
| :--- | :--- | :--- |
| **Feature Selection (Forward)** | **Greedy Algorithm** | At each stage, the algorithm makes the locally optimal choice (the single feature that provides the greatest immediate performance boost). It starts with an empty set and iteratively adds the best available feature. |
| **Performance Metric** | **Mean Absolute Error (MAE)** | Used as the objective function to guide selection. The feature that results in the **lowest MAE** when added to the current feature set is considered the "best" and is permanently added. |
| **Iterative Testing** | **Brute Force Evaluation** | The process involves nested loops. The **Outer Loop** controls the growth of the feature set (the "bucket"). The **Inner Loop** performs a brute-force test, training a new **RandomForest Regressor** instance for *every possible* remaining feature combination at that stage. |
| **Model Instantiation** | **Clean Training State** | **Creating a *new* RF Regressor inside the inner loop** is crucial to ensure that each feature subset test begins from a fresh, untainted initial model state, guaranteeing that performance comparisons are fair and unbiased. |
| **Feature Importance Output** | **Ranked Subset** | The final list of selected feature indices (`idxs_selected_features`) is, by definition of the algorithm, sorted from **most important** (first selected) to **least important** (last selected). |

---

## 🌲 Ex 2 [Bonus]: Comparing External vs. Internal Importance

| ML/Data Science Principle | Concept | Goal |
| :--- | :--- | :--- |
| **Internal Feature Importance** | **Gini/Impurity-Based Ranking** | Extracting the built-in `feature_importances_` attribute from the final **RandomForest Regressor** model. This is an **internal** metric based on how much the feature reduces the Gini Impurity (or MSE for regression) across all trees in the forest. |
| **Comparative Analysis** | **Robustness Check** | Comparing the feature importance order derived from the **Brute Force Forward Selection** (external, performance-based MAE) against the order derived from the **RandomForest's internal mechanism** (impurity-based). |
| **Model Insight** | **Understanding Discrepancies** | Similar rankings confirm the model's internal metric aligns well with actual MAE reduction. Discrepancies might suggest that a feature that locally reduces impurity might not contribute as much to the overall final predictive MAE when combined with other features. |

In [1]:
# Load data
import tarfile
import pandas as pd
import numpy as np
from prettytable import PrettyTable
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import pandas as pd



with tarfile.open("L08_data_inclass.tar.gz", 'r:gz') as tar:
    filename = tar.extractfile("StudentsPerformance.csv")
    if filename:
        df = pd.read_csv(filename)

print('\nOriginal Data:')
display(df.head(5))

# Encode
df_enc = pd.get_dummies(df.drop(columns=['math score']), drop_first=True)

# Convert from boolean to int
columns_bool = df_enc.select_dtypes(include=['bool']).columns
df_enc[columns_bool] = df_enc[columns_bool].astype(int)

print('\nOne Hot Encoded data:')
display(df_enc.head(5))

X = df_enc.values
y = df['math score'].values

classes = np.unique(y)
print('There are %s classes' % len(classes))

table = PrettyTable()
table.title = str('Data shape')
table.field_names = ['X', 'y']
table.add_row([np.shape(X), np.shape(y)])
print(table)


Original Data:


,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75



One Hot Encoded data:


,reading score,writing score,gender_male,race/ethnicity_group B,race/ethnicity_group C,race/ethnicity_group D,race/ethnicity_group E,parental level of education_bachelor's degree,parental level of education_high school,parental level of education_master's degree,parental level of education_some college,parental level of education_some high school,lunch_standard,test preparation course_none
0,72,74,0,1,0,0,0,1,0,0,0,0,1,1
1,90,88,0,0,1,0,0,0,0,0,1,0,1,0
2,95,93,0,1,0,0,0,0,0,1,0,0,1,1
3,57,44,1,0,0,0,0,0,0,0,0,0,0,1
4,78,75,1,0,1,0,0,0,0,0,1,0,1,1


There are 81 classes
+----------------------+
|      Data shape      |
+------------+---------+
|     X      |    y    |
+------------+---------+
| (1000, 14) | (1000,) |
+------------+---------+


In [2]:
def forward_feature_selection(X, y):
    """
    Forward feature selection to identify important features.

    Input:
        X: The feature matrix.
        y: The target variable.

    Output:
        selected_features: List of selected feature indices, sorted by importance.
        mae_list: List of MAE scores corresponding to each selected feature.
    """
    n_features = X.shape[1]
    selected_features = []
    remaining_features = list(range(n_features))

    mae_list = []

    while remaining_features:
        best_mae = np.inf  
        best_feature = None 

        for feature_idx in remaining_features:
            selected_features_iter = selected_features + [feature_idx]
            X_iter = X[:, selected_features_iter]

          
            model = RandomForestRegressor(random_state=42)
            model.fit(X_iter, y)

            
            y_pred = model.predict(X_iter)
            mae = mean_absolute_error(y, y_pred)

            if mae < best_mae:
                best_mae = mae
                best_feature = feature_idx

        if best_feature is not None:
            remaining_features.remove(best_feature)
            selected_features.append(best_feature)
            mae_list.append(best_mae)
        else:
           
            break

    return selected_features, mae_list


In [3]:
selected_features, mae_list = forward_feature_selection(X, y)



feature_names = df_enc.columns.tolist()
selected_feature_names = [feature_names[i] for i in selected_features]

results_df = pd.DataFrame({
    'Feature Index': selected_features,
    'Feature Name': selected_feature_names,
    'MAE': mae_list
})

print("Selected Features Sorted by Importance:")
display(results_df)

Selected Features Sorted by Importance:


,Feature Index,Feature Name,MAE
0,0,reading score,6.766487
1,1,writing score,4.420273
2,2,gender_male,2.680996
3,13,test preparation course_none,2.346228
4,12,lunch_standard,2.135162
5,6,race/ethnicity_group E,2.003529
6,10,parental level of education_some college,1.924233
7,5,race/ethnicity_group D,1.878886
8,8,parental level of education_high school,1.852360
9,7,parental level of education_bachelor's degree,1.840609


In [4]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import pandas as pd

regr = RandomForestRegressor(random_state=42)
regr.fit(X, y)


feature_importances = regr.feature_importances_

#DataFrame 
feature_names = df_enc.columns.tolist()
importance_df = pd.DataFrame({
    'Feature Name': feature_names,
    'Importance': feature_importances
})

# Sorting
importance_df = importance_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

print("Feature Importances from Random Forest:")
display(importance_df)


selected_features, mae_list = forward_feature_selection(X, y)
selected_feature_names = [feature_names[i] for i in selected_features]

forward_selection_df = pd.DataFrame({
    'Selected Features (Forward Selection)': selected_feature_names,
    'MAE': mae_list
})

print("\nFeature Rankings from Forward Selection:")
display(forward_selection_df)

Feature Importances from Random Forest:


,Feature Name,Importance
0,reading score,0.585650
1,writing score,0.215173
2,gender_male,0.119347
3,lunch_standard,0.014609
4,test preparation course_none,0.012707
5,race/ethnicity_group E,0.009987
6,parental level of education_high school,0.006455
7,parental level of education_some college,0.006278
8,race/ethnicity_group C,0.006226
9,race/ethnicity_group D,0.006010



Feature Rankings from Forward Selection:


,Selected Features (Forward Selection),MAE
0,reading score,6.766487
1,writing score,4.420273
2,gender_male,2.680996
3,test preparation course_none,2.346228
4,lunch_standard,2.135162
5,race/ethnicity_group E,2.003529
6,parental level of education_some college,1.924233
7,race/ethnicity_group D,1.878886
8,parental level of education_high school,1.852360
9,parental level of education_bachelor's degree,1.840609


From above  we observe that the sorting of features aligns for the first three features between both methods. After that we have some differences.This is due to the fact that the feature importance in the random forest method is calculated based on the decrease in impurity of the tree nodes. This differs from the forward selection method we used which is based on the decrease in the mean absolute error.
<p>We also observe that the method we used (Forward Selection) overemphasizes features that reduced MAE early and undervaluing those features highly contributive later which makes it biased towards the early features.On the contrary Random Forest  considers features globally and seems to be better for that reason </p>